# 3 — Modeling: connectome-constrained trainable FF/RNN on MNIST

Reconstruct the FlyWire connectome as a **trainable** network whose **connectivity and
synaptic signs are fixed from the data**, but whose **edge magnitudes are learned**, and
classify MNIST.

- 6 subgraphs: `whole`, `whole_right`, `whole_left`, `optic`, `optic_right`, `optic_left`
  (the `optic*` ones include photoreceptors via the stage-2 `visual_mask`).
- 2 flavors: `ff_unroll` (inject at t=0, leak≈1) and `rnn` (persistent inject, leaky, BPTT).
- 2 init modes: `from_data` (magnitudes = scaled real synapse counts) and `random`
  (log-normal matched amplitude — the control).

**Kernel:** `Python (consortium)` — it has torch+CUDA+torchvision+flyconn. (`flyconn_eda`
has no torch.) Heavy training runs via `sbatch slurm/train.sbatch`; this notebook is for
inspection + a short smoke train. Run on a GPU node (`srun`/`sbatch`) for the smoke train.

## Architecture at a glance

**Pixels → photoreceptors → frozen connectome dynamics → biological readout → 10 logits.**
Only the input encoder, the per-edge magnitudes, and the linear readout are learned; the
graph and synaptic signs are fixed from the connectome.

```mermaid
flowchart LR
    IMG["MNIST 784 px"] --> ENC["Encoder<br/>Linear 784→n_input<br/>learned"]
    ENC -->|"into photoreceptors"| CORE
    subgraph CORE["ConnectomeNet (wiring + signs FROZEN; magnitudes LEARNED)"]
        IN["R1-6 / R7 / R8"] -->|"real ± synapses"| HID["subgraph neurons"]
        HID -->|"recurrent/feedback"| HID --> OUT["VPN (+descending)"]
    end
    OUT --> HEAD["mean-pool → Linear →10"] --> L["logits"]
    style CORE fill:#eef6ff,stroke:#4178be
```

Per-edge weight: **`W = sign · softplus(θ)`** — `sign` fixed (Dale's law, never flips),
`θ` the only learned core parameter. Unroll `T` steps of a leaky-`tanh` rate update;
`ff_unroll` injects the image at t=0 (leak≈1), `rnn` injects every step (leaky, BPTT).
See [`README.md`](README.md) for the full diagrams and the per-subgraph size table.

## 1. Build a subgraph and inspect

In [ ]:
import os, numpy as np, torch
os.environ.setdefault('FLYCONN_DATA_ROOT', '/orcd/scratch/orcd/012/mabdel03/connectome_data')
from flyconn.models.subgraphs import build_subgraph, SUBGRAPHS
from flyconn.models import io_inject

print('subgraphs:', list(SUBGRAPHS))
sub = build_subgraph('optic_left')        # smallest visual subgraph
print(f'{sub.subgraph_id}: N={sub.N:,} nodes, E={sub.E:,} edges')
print('sign balance: +', int((sub.sign>0).sum()), ' -', int((sub.sign<0).sum()))
inp = io_inject.input_local_ids(sub); rdo = io_inject.readout_local_ids(sub)
print(f'input (photoreceptor) nodes: {len(inp)}   readout (VPN) nodes: {len(rdo)}')
T, bfs = io_inject.recommend_T(sub, inp, rdo)
print('BFS input->readout:', bfs, '-> recommended T =', T)

## 2. Build the model + apply an init mode

The core is one `ConnectomeNet`: `W_value = sign * softplus(theta)` (sign fixed, magnitude
learned), unrolled `T` steps as a leaky integrator. Spectral radius is rescaled to ~0.9 at
init for stability (no learnable node params).

In [ ]:
from flyconn.models.train import build_classifier
cfg = dict(subgraph_id='optic_left', arch='ff_unroll', init_mode='from_data')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, sub, info = build_classifier(cfg, device=device)
print('device:', device)
print('trainable tensors:', [n for n,p in model.named_parameters() if p.requires_grad][:6], '...')
print('n trainable params:', sum(p.numel() for p in model.parameters() if p.requires_grad))
print('spectral radius after init:', info['init'].get('rho_after'))
# the ONLY core weight DOF is theta (edge magnitudes); encoder + readout are the heads
print('theta (edge magnitudes) shape:', model.core.theta.shape)

## 3. Verification — the four traps

In [ ]:
# (a) gradient reaches theta (no CSR-detach); (b) sign never flips (Dale);
# (c) mask fixed (params == #edges); (d) orientation pre->post.
x = torch.randn(8, sub.N, device=device)
model.train(); logits = model(x); logits.sum().backward()
print('(a) theta.grad nonzero:', float(model.core.theta.grad.abs().sum()) > 0)
w = model.core.edge_weight()
print('(b) sign(W)==fixed sign for all edges:', bool(torch.all(torch.sign(w)==model.core.sign)))
print('(c) #theta == #edges:', model.core.theta.numel()==sub.E)
print('(d) activations finite (stable unroll):', bool(torch.isfinite(model(x)).all()))

## 4. Short smoke train (GPU recommended)

One pass over a few hundred batches to confirm the loss drops and accuracy beats chance
(10%). The full 25-epoch runs go through `sbatch`. Skips quickly on CPU if no GPU.

In [ ]:
from flyconn.models.data import get_loaders
import torch.nn as nn, time
tr, va, te = get_loaders(batch_size=128, num_workers=2, download=True)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()
model.train(); t0=time.time(); losses=[]
MAXB = 200 if device=='cuda' else 20      # keep CPU quick
for b,(xb,yb) in enumerate(tr):
    if b>=MAXB: break
    xb,yb = xb.to(device), yb.to(device)
    opt.zero_grad(); loss=crit(model(xb), yb); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    losses.append(float(loss))
print(f'trained {len(losses)} batches in {time.time()-t0:.1f}s on {device}; '
      f'loss {losses[0]:.3f} -> {losses[-1]:.3f}')

In [ ]:
import matplotlib.pyplot as plt
plt.plot(losses); plt.xlabel('batch'); plt.ylabel('train loss')
plt.title('smoke-train loss (optic_left / ff_unroll / from_data)'); plt.show()

# quick val accuracy on a few batches
model.eval(); correct=total=0
with torch.no_grad():
    for b,(xb,yb) in enumerate(va):
        if b>=20: break
        xb,yb=xb.to(device),yb.to(device)
        correct+=int((model(xb).argmax(1)==yb).sum()); total+=yb.numel()
print(f'val acc on {total} samples: {correct/total:.3f}  (chance=0.10)')

## 5. Launch the full grid

The 24-run grid (6 subgraphs × {ff_unroll, rnn} × {from_data, random}) trains via a SLURM
array on `pi_tpoggio`:

```bash
cd "3 - Modeling" && ./03_train.sh          # sbatch ../slurm/train_array.sbatch (--array=0-23%4)
# or one run:
EXP=optic_left_ff_unroll_initA sbatch slurm/train.sbatch
```

Results land on scratch: `$FLYCONN_DATA_ROOT/v783/models/<run>/{ckpt_best.pt,metrics.jsonl,summary.json}`.
The headline comparison is **from_data vs random** test accuracy per subgraph — does the real
wiring beat amplitude-matched random wiring?